# Compare models

In [2]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

# --- Config ---
cwd = Path.cwd().resolve()
if (cwd / "notebooks").is_dir():
    project_root = cwd
else:
    notebooks_dir = next((path for path in [cwd, *cwd.parents] if path.name == "notebooks"), None)
    if notebooks_dir is None:
        raise ValueError("Could not locate the project root from the notebook working directory.")
    project_root = notebooks_dir.parent

results_root = project_root / "results"
timestamp = "Path_2025-03-20_19-00-00"  # March 20 at 19:00
run_dir = results_root / timestamp


def extract_param_count(model_name: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)([KMBT])(?=-|$)", model_name, re.IGNORECASE)
    if not match:
        return float("inf")

    value = float(match.group(1))
    unit = match.group(2).upper()
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9, "T": 1e12}
    return value * multipliers[unit]


# --- Collect metrics for each model ---
# Structure target:
# columns = MultiIndex [model, {with image, without image}]
# rows    = [structure adherence pct, task accuracy pct]
records = {}

for vendor_dir in sorted(run_dir.iterdir()):
    if not vendor_dir.is_dir():
        continue
    for model_dir in sorted(vendor_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        with_path = model_dir / "with_image.json"
        without_path = model_dir / "without_image.json"

        if not with_path.exists() or not without_path.exists():
            continue

        with with_path.open("r", encoding="utf-8") as f:
            with_data = json.load(f)
        with without_path.open("r", encoding="utf-8") as f:
            without_data = json.load(f)

        records[model_name] = {
            ("with image", "structure adherence pct"): with_data["overall_summary"]["structure_adherence_pct"],
            ("with image", "task accuracy pct"): with_data["overall_summary"]["task_accuracy_pct"],
            ("without image", "structure adherence pct"): without_data["overall_summary"]["structure_adherence_pct"],
            ("without image", "task accuracy pct"): without_data["overall_summary"]["task_accuracy_pct"],
        }

if not records:
    raise ValueError(f"No valid model result pairs found in: {run_dir}")

# --- Build table ---
row_order = ["structure adherence pct", "task accuracy pct"]
col_order = ["with image", "without image"]

data = {}
for model_name, vals in records.items():
    data[(model_name, "with image")] = [
        vals[("with image", "structure adherence pct")],
        vals[("with image", "task accuracy pct")],
    ]
    data[(model_name, "without image")] = [
        vals[("without image", "structure adherence pct")],
        vals[("without image", "task accuracy pct")],
    ]

table = pd.DataFrame(data, index=row_order)

# Sort models by parameter count, then by name, and enforce subcolumn order.
models_sorted = sorted(records.keys(), key=lambda name: (extract_param_count(name), name.lower()))
table = table.reindex(columns=pd.MultiIndex.from_product([models_sorted, col_order]))

# --- Format and display ---
styled = (
    table.style
    .format("{:.1f}%")
    .set_caption(f"Model Results ({timestamp})")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ])
)

display(styled)

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

# --- Customization ---
run_id = "Steps_2026-03-24_21-59-08"
vendor_name = "Qwen"
model_name = "Qwen2.5-VL-32B-Instruct"
image_mode = "without_image"  # "with_image" or "without_image"

prompt_order = [f"p{i}" for i in range(5)]
schema_order = [f"st{i}" for i in range(4)]
heatmap_cmap = "YlGnBu"

# --- Resolve project root ---
cwd = Path.cwd().resolve()
if (cwd / "notebooks").is_dir():
    project_root = cwd
else:
    notebooks_dir = next((path for path in [cwd, *cwd.parents] if path.name == "notebooks"), None)
    if notebooks_dir is None:
        raise ValueError("Could not locate the project root from the notebook working directory.")
    project_root = notebooks_dir.parent

results_dir = project_root / "results" / run_id / vendor_name / model_name / image_mode
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")

# --- Load one row per prompt/schema config ---
rows = []
for path in sorted(results_dir.glob("*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    runs = payload["runs"]
    total_runs = len(runs)
    completion_count = sum(bool(run.get("completion")) for run in runs)
    perfect_structure_count = sum(bool(run.get("perfect_structure")) for run in runs)

    rows.append(
        {
            "prompt_id": payload["userprompt_id"],
            "schema_id": payload["schema_id"],
            "config_id": f"{payload['userprompt_id']}_{payload['schema_id']}",
            "total_runs": total_runs,
            "completion_count": completion_count,
            "completion_rate": 100 * completion_count / total_runs if total_runs else 0.0,
            "perfect_structure_count": perfect_structure_count,
            "perfect_structure_rate": 100 * perfect_structure_count / total_runs if total_runs else 0.0,
        }
    )

config_df = pd.DataFrame(rows)
if config_df.empty:
    raise ValueError(f"No result files found in: {results_dir}")

if prompt_order is None:
    prompt_order = sorted(config_df["prompt_id"].unique())
if schema_order is None:
    schema_order = sorted(config_df["schema_id"].unique())

config_df["prompt_id"] = pd.Categorical(config_df["prompt_id"], categories=prompt_order, ordered=True)
config_df["schema_id"] = pd.Categorical(config_df["schema_id"], categories=schema_order, ordered=True)
config_df = config_df.sort_values(["prompt_id", "schema_id"]).reset_index(drop=True)
config_df["completion_label"] = config_df["completion_count"].astype(str) + "/" + config_df["total_runs"].astype(str)

heatmap_df = (
    config_df
    .pivot(index="prompt_id", columns="schema_id", values="completion_rate")
    .reindex(index=prompt_order, columns=schema_order)
)

schema_summary = (
    config_df
    .groupby("schema_id", observed=False, as_index=False)
    .agg(
        prompts_with_any_success=("completion_rate", lambda values: int((values > 0).sum())),
        mean_completion_rate=("completion_rate", "mean"),
        max_completion_rate=("completion_rate", "max"),
    )
    .sort_values(["prompts_with_any_success", "mean_completion_rate", "schema_id"], ascending=[False, False, True])
    .reset_index(drop=True)
)

prompt_summary = (
    config_df
    .groupby("prompt_id", observed=False, as_index=False)
    .agg(
        schemas_with_any_success=("completion_rate", lambda values: int((values > 0).sum())),
        mean_completion_rate=("completion_rate", "mean"),
        max_completion_rate=("completion_rate", "max"),
    )
    .sort_values(["schemas_with_any_success", "mean_completion_rate", "prompt_id"], ascending=[False, False, True])
    .reset_index(drop=True)
)

nonzero_configs = (
    config_df[config_df["completion_rate"] > 0]
    .sort_values(["completion_rate", "completion_count", "config_id"], ascending=[True, True, True])
    .reset_index(drop=True)
)

print(f"Loaded {len(config_df)} configs from {results_dir}")
display(config_df)

In [ ]:
plt.figure(figsize=(8, 5))
annot_df = heatmap_df.apply(lambda col: col.map(lambda value: "" if pd.isna(value) else f"{value:.0f}%"))

sns.heatmap(
    heatmap_df,
    annot=annot_df,
    fmt="",
    cmap=heatmap_cmap,
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "Task completion rate (%)"},
)
plt.title(f"Completion Heatmap: {model_name} ({image_mode.replace('_', ' ')})")
plt.xlabel("Schema")
plt.ylabel("Prompt")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = sns.color_palette("Blues", n_colors=len(schema_summary))
bars = ax.bar(schema_summary["schema_id"].astype(str), schema_summary["prompts_with_any_success"], color=colors)

ax.set_title(f"Schemas That Work Across Prompts: {model_name}")
ax.set_xlabel("Schema")
ax.set_ylabel("Prompts with nonzero completion")
ax.set_ylim(0, max(schema_summary["prompts_with_any_success"].max() + 0.8, 1.5))

for bar, (_, row) in zip(bars, schema_summary.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"avg {row['mean_completion_rate']:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

display(schema_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = sns.color_palette("Greens", n_colors=len(prompt_summary))
bars = ax.bar(prompt_summary["prompt_id"].astype(str), prompt_summary["schemas_with_any_success"], color=colors)

ax.set_title(f"Prompts That Work Across Schemas: {model_name}")
ax.set_xlabel("Prompt")
ax.set_ylabel("Schemas with nonzero completion")
ax.set_ylim(0, max(prompt_summary["schemas_with_any_success"].max() + 0.8, 1.5))

for bar, (_, row) in zip(bars, prompt_summary.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"avg {row['mean_completion_rate']:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

display(prompt_summary)

In [ ]:
if nonzero_configs.empty:
    print("No configs with nonzero completion rate were found for the selected slice.")
else:
    fig, ax = plt.subplots(figsize=(8, max(3, 0.6 * len(nonzero_configs))))

    ax.hlines(
        y=nonzero_configs["config_id"],
        xmin=0,
        xmax=nonzero_configs["completion_rate"],
        color="0.75",
        linewidth=2,
    )
    ax.scatter(
        nonzero_configs["completion_rate"],
        nonzero_configs["config_id"],
        s=100,
        color=sns.color_palette("rocket", n_colors=1)[0],
        zorder=3,
    )

    for _, row in nonzero_configs.iterrows():
        ax.text(
            row["completion_rate"] + 1,
            row["config_id"],
            f"{row['completion_count']}/{row['total_runs']} ({row['completion_rate']:.0f}%)",
            va="center",
            fontsize=9,
        )

    ax.set_title(f"Nonzero-Completion Configs: {model_name} ({image_mode.replace('_', ' ')})")
    ax.set_xlabel("Task completion rate (%)")
    ax.set_ylabel("Prompt + schema config")
    ax.set_xlim(0, min(115, nonzero_configs["completion_rate"].max() + 15))
    plt.tight_layout()
    plt.show()

    display(nonzero_configs[["config_id", "completion_count", "total_runs", "completion_rate"]])